In [ ]:
import numpy as np

# ── 1. 기초 데이터 ─────────────────────────────────────────────
CROPS    = ['양상추','토마토','대파','오이','고추','시금치','무','배추','옥수수']
N_CROP   = 9
N_PERIOD = 24   # 1월상(0) ~ 12월하(23), 1반기 = 15일

# 1m² 파종 시 필요한 면적 비율 (가격->단위수익 환산에 사용)
area_per = np.array([0.10, 0.30, 0.15, 0.40, 0.25, 0.05, 0.20, 0.20, 0.20])

# 성장 기간 (반기 단위): 노지 / 하우스
grow_O = np.array([ 4,  8,  6,  5, 10,  3,  6,  5,  6])
grow_H = np.array([ 3,  6,  5,  4,  8,  2,  5,  4,  5])

# 월별 시장 가격 (0 = 해당 월 판매 불가)
price_O_monthly = np.array([
    [2500,2500,2500,2500,2500,   0,   0,   0,   0,   0,2200,2200],
    [   0,   0,   0,   0,15000,18000,18000,15000,12000,  0,  0,  0],
    [   0,   0,4000,4000,4500,5000,5000,5500,5000,4500,  0,  0],
    [   0,   0,   0,   0,11000,11000,13000,15000,13000,11000,0, 0],
    [   0,   0,   0,   0,   0,   0,10000,12000,12000,10000,  0,  0],
    [ 800, 800, 600, 600,   0,   0,   0,   0, 700, 700, 800, 800],
    [   0,   0,   0,   0,   0,   0,   0,   0,4500,4800,4800,4800],
    [   0,   0,   0,   0,   0,   0,   0,   0,3200,3200,3600,3600],
    [   0,   0,   0,   0,   0,   0,3000,3500,3000,   0,   0,   0],
])
price_H_monthly = np.array([
    [3500,3500,3200,3200,3200,3000,3000,3500,3500,3200,3200,3200],
    [   0,   0,25000,25000,21000,25000,25000,21000,18000,20000,22000,22000],
    [6000,6000,6000,6000,6500,7000,7000,8000,7000,6500,7000,7000],
    [   0,   0,15000,15000,15000,16000,18000,21000,18000,16000,18000,18000],
    [   0,   0,15000,15000,15000,15000,14000,17000,17000,15000,18000,18000],
    [1200,1200,1000,1000,1000,1000,1000,1200,1000,1000,1200,1200],
    [6500,6500,6000,6000,6000,6000,6000,6500,6000,6500,6500,6500],
    [5500,5500,5000,5000,5000,5000,5000,5500,4800,4800,5000,5000],
    [   0,   0,   0,   0,4500,4500,4200,5000,4200,4500,   0,   0],
])

# 2. 면적 제약 (명세: 구역 면적의 5% 이상 25% 이하, 연속값)
AREA_O = 1200.0
AREA_H =  300.0
MIN_O  = AREA_O * 0.05   # 60.0  m² (하한선, 단위 아님)
MAX_O  = AREA_O * 0.25   # 300.0 m²
MIN_H  = AREA_H * 0.05   # 15.0  m²
MAX_H  = AREA_H * 0.25   # 75.0  m²

# 3. 유효 슬롯 필터링 및 단위 수익 계산
valid_O = np.zeros((N_CROP, N_PERIOD), dtype=bool)
valid_H = np.zeros((N_CROP, N_PERIOD), dtype=bool)
rev_O   = np.zeros((N_CROP, N_PERIOD))
rev_H   = np.zeros((N_CROP, N_PERIOD))

for i in range(N_CROP):
    for t in range(N_PERIOD):
        ht_O = t + grow_O[i] - 1
        ht_H = t + grow_H[i] - 1
        if ht_O < N_PERIOD:
            p = price_O_monthly[i, ht_O // 2]
            if p > 0:
                valid_O[i, t] = True
                rev_O[i, t]   = p / area_per[i]
        if ht_H < N_PERIOD:
            p = price_H_monthly[i, ht_H // 2]
            if p > 0:
                valid_H[i, t] = True
                rev_H[i, t]   = p / area_per[i]

# 4. 반기별 최적 배정 함수 (연속값 그리디)
def solve_period(candidates, total_area, occupied, t, min_a, max_a):
    """
    t 반기의 후보 작물들에 대해 연속 면적을 최적 배정.
    수익 = 단위수익 x 면적 (선형)이므로,
    성장 구간 전체(t ~ t+grow-1)의 잔여 면적 내에서
    수익 높은 순으로 최대 면적을 배정하는 그리디가 최적해.
    제약: min_a <= 배정 면적 <= max_a (연속값, 이산화 없음)
    """
    if not candidates:
        return 0.0, {}

    # 정렬: 1) 단위수익 내림  2) 성장기간 오름 (땅 덜 묶임)  3) 작물번호 오름 (재현성)
    candidates = sorted(candidates, key=lambda x: (-x[0], x[2], x[1]))

    last_p = max(t + g - 1 for _, _, g in candidates)
    cap = [total_area - occupied[p] for p in range(t, last_p + 1)]

    alloc  = {}
    profit = 0.0

    for rv, ci, grow in candidates:
        feasible = max_a
        for r in range(grow):
            feasible = min(feasible, cap[r])

        if feasible < min_a:
            continue

        area = feasible
        alloc[ci] = area
        profit += rv * area
        for r in range(grow):
            cap[r] -= area

    return profit, alloc

# 5. 메인 루프: 반기별 순서대로 배정
xO = np.zeros((N_CROP, N_PERIOD))
xH = np.zeros((N_CROP, N_PERIOD))
occupied_O = np.zeros(N_PERIOD)
occupied_H = np.zeros(N_PERIOD)

for t in range(N_PERIOD):
    cands_O = [(rev_O[i,t], i, grow_O[i]) for i in range(N_CROP) if valid_O[i,t]]
    if cands_O:
        _, alloc_O = solve_period(cands_O, AREA_O, occupied_O, t, MIN_O, MAX_O)
        for ci, area in alloc_O.items():
            xO[ci, t] = area
            for p in range(t, t + grow_O[ci]):
                occupied_O[p] += area

    cands_H = [(rev_H[i,t], i, grow_H[i]) for i in range(N_CROP) if valid_H[i,t]]
    if cands_H:
        _, alloc_H = solve_period(cands_H, AREA_H, occupied_H, t, MIN_H, MAX_H)
        for ci, area in alloc_H.items():
            xH[ci, t] = area
            for p in range(t, t + grow_H[ci]):
                occupied_H[p] += area

# 6. 수익 계산
crop_rev  = np.array([(rev_O[i]*xO[i] + rev_H[i]*xH[i]).sum() for i in range(N_CROP)])
total_rev = crop_rev.sum()

# 7. 결과 출력
print("=" * 56)
print(f"  수정 DP 해 (연속 면적): {total_rev/1e6:.4f} 백만원")
print("=" * 56)
print(f"\n{'작물':<6} {'수익(백만원)':>12} {'노지 파종합(m2)':>14} {'하우스 파종합(m2)':>16}")
print("-" * 52)
for i in range(N_CROP):
    print(f"{CROPS[i]:<6} {crop_rev[i]/1e6:>12.3f}"
          f" {xO[i].sum():>14.1f} {xH[i].sum():>16.1f}")
print("-" * 52)
print(f"{'합계':<6} {total_rev/1e6:>12.4f}"
      f" {xO.sum():>14.1f} {xH.sum():>16.1f}")

print("\n[제약 확인]")
print(f"노지   최대 점유: {occupied_O.max():.1f} / {AREA_O:.1f} m2")
print(f"하우스 최대 점유: {occupied_H.max():.1f} / {AREA_H:.1f} m2")
bad_O = [p for p in range(N_PERIOD) if occupied_O[p] > AREA_O + 1e-6]
bad_H = [p for p in range(N_PERIOD) if occupied_H[p] > AREA_H + 1e-6]
if not bad_O and not bad_H:
    print("모든 반기에서 면적 제약 만족.")
else:
    if bad_O: print("노지 위반 반기:", bad_O)
    if bad_H: print("하우스 위반 반기:", bad_H)

# 8. 반기별 상세 현황 (명세 출력 형식: 노지m2/하우스m2)
print("\n[반기별 파종 현황 - 노지m2/하우스m2]")
print("-" * 90)
print(f"  {'반기':<5} | " + " | ".join(f"{c:<7}" for c in CROPS))
print("-" * 90)
for t in range(N_PERIOD):
    m = t // 2 + 1
    h = '상' if t % 2 == 0 else '하'
    cells = []
    for i in range(N_CROP):
        o, hh = xO[i,t], xH[i,t]
        cells.append(f"{o:.0f}/{hh:.0f}" if (o > 0 or hh > 0) else "   -   ")
    print(f"  {m:>2}월{h} | " + " | ".join(f"{c:<7}" for c in cells))